# Huấn luyện chatbot học vụ trên Colab

Notebook này chạy đúng đường ống của dự án, không phải bản rút gọn: cùng dataset, cùng siêu
tham số, cùng khâu đánh giá và benchmark. Chỉ đổi máy chạy.

**Chạy các cell theo đúng thứ tự.** Cell *lượt ngắn* ở giữa là chốt quan trọng nhất — nó tốn
vài phút và cho biết ngay TPU có chạy được và có nhanh không, thay vì để bạn phát hiện sau một
tiếng.

### Bốn điều cần biết trước

**Runtime TPU của Colab chỉ có sẵn JAX, không có `torch_xla`.** Thiếu nó thì PyTorch không thấy
TPU và lặng lẽ chạy bằng CPU — chậm tới mức không dùng được, mà **không báo lỗi gì**. Cell 2b
cài nó, và sau khi cài **phải khởi động lại runtime**.

**TPU ở đây dùng một lõi.** Dùng cả 8 lõi phải viết lại khâu đánh giá cho chạy song song, mà
khâu đó có bước chạy truy vấn trên ontology nên không song song hoá dễ. Một lõi là thứ chạy
được ngay, không phải sửa gì.

**Huấn luyện chạy ngay trong notebook**, không gọi ra tiến trình riêng. TPU chỉ cho một tiến
trình chiếm giữ, mà cell kiểm tra bên dưới đã chiếm nó rồi.

**Nếu TPU chậm hơn máy bạn thì đổi runtime sang GPU và chạy lại từ cell 1** — mã tự nhận diện
máy tăng tốc, không phải sửa dòng nào. Đọc cell cuối notebook.

## 1. Khai báo môi trường rồi xem Colab cấp máy gì

Các biến môi trường phải đặt **trước khi `torch` được nạp**, nên đây phải là cell chạy đầu
tiên. Nếu bạn đã chạy cell khác trước, hãy khởi động lại runtime.

In [ ]:
import os

# Phải đặt trước khi torch/torch_xla khởi tạo, không thì bị bỏ qua âm thầm.
os.environ.setdefault("PJRT_DEVICE", "TPU")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# TPU chỉ cho MỘT thư viện chiếm giữ. Nếu jax kịp giữ trước thì torch_xla không
# lấy được nữa, nên đừng để thư viện nào đánh thức jax.
os.environ.setdefault("USE_JAX", "0")
os.environ.setdefault("USE_FLAX", "0")

import platform
import sys

print("Python:", platform.python_version())

import torch

print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có")

try:
    import torch_xla
    import torch_xla.runtime as xr

    print("torch_xla:", torch_xla.__version__)
    print("lõi TPU thấy được:", xr.global_runtime_device_count())
except Exception as exc:
    print("torch_xla: KHÔNG dùng được ->", type(exc).__name__, exc)
    print("=> muốn dùng TPU thì chạy cell 2b; nếu không thì sẽ rơi xuống CPU")

## 2. Cài thư viện

`torch` được **ghim về đúng bản Colab đang có**. Nếu để pip tự do, nó có thể kéo một bản torch
khác về và làm lệch cặp `torch`/`torch_xla` — lỗi đó biểu hiện muộn và rất khó truy.

Dự án **không cài thành gói**, chỉ cần nằm trong đường dẫn tìm kiếm, nên khỏi lệ thuộc việc
Colab dùng Python bản nào.

In [ ]:
import pathlib
import subprocess

pathlib.Path("/content/constraints.txt").write_text("torch==" + torch.__version__ + "\n")

packages = [
    "transformers>=5.14.1",
    "peft>=0.20.0",
    "accelerate>=1.14.0",
    "datasets>=5.0.0",
    "sentencepiece>=0.2.2",
    "rdflib>=7.6.0",
    "owlrl>=7.6.2",
    "huggingface_hub>=1.24.0",
]
if torch.cuda.is_available():
    # adamw_8bit là bộ tối ưu của bitsandbytes, chỉ chạy trên CUDA. Trên TPU thì
    # không cần cài, vì mã đã tự đổi sang bộ tối ưu khác.
    packages.append("bitsandbytes>=0.49.2")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", "/content/constraints.txt"] + packages,
    check=True,
)
print("đã cài xong")

## 2b. Cài `torch_xla` — chỉ khi dùng TPU

**Bỏ qua cell này nếu bạn đang dùng GPU.**

Không có `torch_xla` thì PyTorch không thấy TPU và tụt xuống CPU mà không báo gì. Dấu hiệu nhận
ra: dòng cảnh báo `pin_memory ... no accelerator is found`, và tốc độ khoảng 0,1 mẫu mỗi giây.

Sau khi cell này cài xong, phải **`Runtime → Restart session`**, rồi chạy lại từ cell 1 và bỏ
qua chính cell này. Thư viện đã cài vẫn còn sau khi khởi động lại.

In [ ]:
try:
    import torch_xla

    print("torch_xla đã có:", torch_xla.__version__)
except ImportError:
    version = torch.__version__.split("+")[0]
    print("chưa có torch_xla; đang cài bản khớp torch", version)
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "torch_xla[tpu]~=" + version,
            "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
        ],
        check=True,
    )
    print()
    print("ĐÃ CÀI XONG. Bây giờ: Runtime -> Restart session,")
    print("rồi chạy lại từ cell 1 và BỎ QUA cell 2b này.")

## 3. Đăng nhập Hugging Face

T5Gemma bị khoá theo giấy phép Gemma. Token phải thuộc tài khoản **đã bấm đồng ý giấy phép**
trên trang model, nếu không bước tải model báo lỗi 401 hoặc 403.

In [ ]:
from huggingface_hub import login

login()

## 4. Lấy mã nguồn và dữ liệu

In [ ]:
REPO = "https://github.com/vpthinh19/ontology-chatbot.git"
BRANCH = "ontology-v2"
WORK = "/content/ontology-chatbot"

if os.path.isdir(os.path.join(WORK, ".git")):
    subprocess.run(["git", "-C", WORK, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", WORK, "reset", "--hard", "origin/" + BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORK], check=True)

os.chdir(WORK)
if WORK + "/src" not in sys.path:
    sys.path.insert(0, WORK + "/src")

print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# Python giữ module đã nạp trong bộ nhớ. Nếu phiên này từng nạp mã cũ thì việc
# vừa tải bản mới về KHÔNG có tác dụng nào cả.
stale = [name for name in sys.modules if name.startswith("ontchatbot")]
if stale:
    print("CHÚ Ý: mã cũ đang nằm trong bộ nhớ của phiên này.")
    print("Phải Runtime -> Restart session rồi chạy lại từ cell 1,")
    print("nếu không thì bản vừa tải về sẽ không được dùng.")

## 5. Kiểm dữ liệu đã lên đủ chưa

In [ ]:
from ontchatbot.research.training import MAX_TARGET_LENGTH, MODEL_SPECS
from ontchatbot.settings import DATASET_DIR

for split in ("train", "val", "test"):
    path = DATASET_DIR / (split + ".jsonl")
    print(split.ljust(6), sum(1 for _ in path.open(encoding="utf-8")), "dòng")

spec = MODEL_SPECS["t5gemma2"]
print("trần sinh:", MAX_TARGET_LENGTH, "· lô:", spec["batch_size"], "· lô đánh giá:", spec["eval_batch_size"])

## 6. Lượt ngắn — chốt quan trọng nhất

Đừng bỏ qua cell này. Lượt ngắn cố ý lấy **16 đích dài nhất** tập train và 8 đích dài nhất tập
val, nên nó là phép thử bộ nhớ ở trường hợp xấu nhất. Qua được lượt ngắn thì lượt thật không
tràn.

Nó cũng trả lời câu hỏi thật sự quan trọng: **TPU ở đây nhanh hay chậm.** Con số thời gian in
ra ở cuối là căn cứ để quyết định chạy tiếp trên TPU hay đổi sang GPU.

Lần chạy đầu còn phải tải model từ Hugging Face, nên có thêm vài phút không liên quan tới tốc
độ tính toán.

In [ ]:
import gc
import inspect
import shutil
import time

from ontchatbot.research.training import (
    _accelerator_kind,
    _generate_rows,
    _parse_args,
    train,
)

# Kiểm mã đang chạy có đúng là bản mới không. Module đã nạp thì nằm luôn trong
# bộ nhớ, nên tải bản mới về giữa phiên là vô ích - và biểu hiện của nó là lỗi
# "Cannot set version_counter for inference tensor" ở khâu sinh câu trả lời,
# tức là mất trọn một lượt huấn luyện mới biết.
if "no_grad" not in inspect.getsource(_generate_rows):
    raise SystemExit(
        "Đang chạy mã CŨ nạp sẵn trong bộ nhớ phiên này. "
        "Runtime -> Restart session, rồi chạy lại từ cell 1."
    )

# Chặn cứng ở đây. Một lượt thật trên CPU mất nhiều giờ, và điều tệ nhất là nó
# KHÔNG báo lỗi - cứ chạy, chỉ chậm.
accelerator = _accelerator_kind(torch)
print("máy tăng tốc:", accelerator)
if accelerator == "cpu":
    raise SystemExit(
        "Đang chạy bằng CPU, đừng huấn luyện. Chọn một trong hai: "
        "chạy cell 2b để cài torch_xla rồi KHỞI ĐỘNG LẠI runtime, "
        "hoặc đổi runtime sang GPU rồi chạy lại từ cell 1."
    )

shutil.rmtree(WORK + "/artifacts/smoke", ignore_errors=True)

start = time.monotonic()
smoke_report = train(
    _parse_args(
        [
            "--model", "t5gemma2",
            "--output-dir", WORK + "/artifacts/smoke",
            "--seed", "42",
            "--smoke-test",
            "--no-gradient-checkpointing",
        ]
    )
)
print()
print("lượt ngắn xong sau", round(time.monotonic() - start), "giây")

# Nhả bộ nhớ trước lượt thật, vì cả hai chạy trong cùng tiến trình.
del smoke_report
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Gắn Google Drive và đặt tên lượt chạy

Kết quả ghi **thẳng vào Drive**, không ghi vào ổ tạm của Colab. Lượt 9 trên máy ở nhà đã chết
đúng ở khâu ghi model và mất trọng số; trên Colab rủi ro đó cao hơn vì phiên có thể bị ngắt.
Các điểm lưu giữa chừng chỉ là bộ chỉnh LoRA nên nhẹ, ghi lên Drive không tốn thời gian.

**Đổi `RUN_NAME` ở đây, chỉ một chỗ này.** Thư mục đầu ra bắt buộc phải trống, nếu không script
dừng ngay từ đầu.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "run10-colab"
OUT = "/content/drive/MyDrive/ontology-chatbot/" + RUN_NAME

print("kết quả sẽ ghi vào:", OUT)
print("đã tồn tại (phải trống mới chạy được):", os.path.isdir(OUT + "/t5gemma2"))

## 8. Lượt huấn luyện thật

Mọi siêu tham số **giữ y nguyên** như chín lượt trước: lô 8, hạt giống 42, 16 epoch, đánh giá
mỗi 4 epoch. Chỉ tắt chế độ tiết kiệm bộ nhớ — việc đó không đổi phép tính, chỉ nhanh hơn — nên
lượt này vẫn so thẳng được với lượt 9.

Một lệnh làm cả ba việc: huấn luyện, ghi model, rồi chạy benchmark.

In [ ]:
start = time.monotonic()
report = train(
    _parse_args(
        [
            "--model", "t5gemma2",
            "--output-dir", OUT,
            "--seed", "42",
            "--save-model",
            "--benchmark-after-training",
            "--no-gradient-checkpointing",
        ]
    )
)
print()
print("lượt", RUN_NAME, "xong sau", round((time.monotonic() - start) / 60, 1), "phút")
print("kết quả ở:", OUT)

## 9. Đọc kết quả

Chỉ so thẳng được với lượt 9 khi **số câu chấm bằng 349**. Nếu khác thì tập chấm đã đổi, phải
quy về tỉ lệ trước khi kết luận bất cứ điều gì — đây là cái bẫy đã dính nhiều lần ở dự án này.

In [ ]:
import json
from pathlib import Path

result_dir = Path(OUT) / "t5gemma2"
saved = json.loads((result_dir / "metrics.json").read_text(encoding="utf-8"))
overall = saved["overall"]
training = saved["training"]

print("số câu chấm:", overall["count"], "(lượt 7, 8, 9 đều chấm 349 câu)")
print("Answer Exact      {:.1%}   (lượt 7: 82,2% · lượt 8: 75,1% · lượt 9: 75,1%)".format(overall["answer_exact_rate"]))
print("Hệ thống trả đúng {:.1%}   (lượt 7: 85,0% · lượt 8: 79,4%)".format(overall["system_answer_exact_rate"]))
print("Từ chối an toàn   {:.1%}   (lượt 7: 74,4% · lượt 8: 80,0% · lượt 9: 52,2%)".format(overall["safe_rejection_rate"]))
print()
print("máy chạy:", training["accelerator"], "·", training["dtype"], "· bộ tối ưu:", training["optimizer"])
print("tiết kiệm bộ nhớ:", training["gradient_checkpointing"])
print("thời gian huấn luyện:", round(training["train_runtime_seconds"] / 60, 1), "phút")
print("mất mát cuối:", training["train_loss"])

benchmark_path = result_dir / "benchmark_metrics.json"
if benchmark_path.is_file():
    benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
    print()
    print("benchmark:", json.dumps(benchmark["overall"], ensure_ascii=False, indent=2))

## 10. Chỉ số từ chối, soi theo từng nhóm

Con số tổng che mất chuyện quan trọng. Nhóm `greeting-social` đạt 0/n là **đúng ý đồ**, không
phải hỏng: câu chào có đích là truy vấn liệt kê năng lực chứ không phải "không có thông tin".

Và trước khi kết luận model kém ở nhóm nào, phải nhớ mọi nhóm từ chối **chỉ được chấm bằng
những lối nói chưa từng được dạy**. Đã bốn lần ở dự án này một con số thấp hoá ra chỉ là "mẫu
chưa từng dạy".

Đây cũng là việc đang treo: chỉ số từ chối tụt xuống 52,2% ở lượt 9 mà chưa ai truy ra vì sao.

In [ ]:
from ontchatbot.settings import REJECTION_CHECKLIST_PATH

checklist = json.loads(REJECTION_CHECKLIST_PATH.read_text(encoding="utf-8"))
cases = {case["id"]: case for case in saved["cases"]}

for group in sorted(checklist):
    ids = [case_id for case_id in checklist[group] if case_id in cases]
    if not ids:
        continue
    passed = sum(1 for case_id in ids if cases[case_id].get("safe_rejection"))
    print(group.ljust(22), str(passed) + "/" + str(len(ids)))

## Nếu TPU không đủ nhanh, hoặc `torch_xla` không lên được

Đổi runtime sang GPU (`Runtime → Change runtime type → GPU`), **khởi động lại runtime**, rồi
chạy lại từ cell 1. Mã tự nhận diện máy tăng tốc: nó tự đổi độ chính xác số học và bộ tối ưu
cho phù hợp, bạn không phải sửa dòng nào.

Đừng cố quá lâu với `torch_xla`. Nếu cài xong và khởi động lại mà cell 1 vẫn không thấy lõi TPU
nào, khả năng cao là JAX đã giữ TPU trước — đổi sang GPU nhanh hơn là gỡ tiếp.

**Một cảnh báo về GPU của bản Colab miễn phí.** Card T4 không hỗ trợ định dạng số `bfloat16`,
nên mã sẽ tụt xuống `float16`. Dòng model T5 nổi tiếng dễ tràn số ở `float16` — nếu bạn thấy
mất mát thành `nan` ngay những bước đầu thì đó là nguyên nhân, không phải dữ liệu hay siêu tham
số.

Điểm này cũng đáng đưa vào báo cáo: **TPU tính bằng `bfloat16`, giống đúng card RTX 4050 ở
nhà**, nên con số đo trên TPU so được với chín lượt cũ; con số đo trên T4 thì không.

### Muốn chạy tiếp hai model còn lại

Quy trình cuối của dự án là huấn luyện cả ba model rồi so, lấy con tốt nhất đem deploy. Đổi
`--model t5gemma2` thành `vit5` hoặc `bartpho`, và **đổi cả `RUN_NAME`**. Với ViT5 thì lần đầu
chạy sẽ tự vá từ điển trước, mất thêm ít phút.